In [55]:
import io
import json
import os
import pdal
import richdem as rd
import numpy as np
import json
import laspy
from pyproj import CRS
from config import *
from rasterio.io import MemoryFile
from rasterio.merge import merge
from estimation_nodata import fill_nodata_knn_chunkwise
import rasterio
from pathlib import Path
from osgeo import gdal
from sklearn.neighbors import KNeighborsRegressor
from whitebox.whitebox_tools import WhiteboxTools

In [56]:
data_dir = '/home/ajai-krishna/work/GEO_AI/data/'
out_dir = '/home/ajai-krishna/work/GEO_AI/outputs/subsets'
output_training_tif = '/home/ajai-krishna/work/GEO_AI/data/input/subsets'

In [57]:

crs = CRS.from_proj4("+proj=utm +zone=43 +datum=WGS84 +units=m +no_defs")

In [58]:
gujarath_data = '/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las'
# gujarath_las = laspy.open(gujarath_data_path)

In [59]:
# with laspy.open(gujarath_data) as gujarath_las:
#     gujarath_las_header = gujarath_las.header
#     print(list(gujarath_las_header.point_format.dimension_names))
#     las_x = gujarath_las.X
#     las_y = gujarath_las.Y
#     las_z = gujarath_las.Z
#     print(las_z)

In [60]:
# gujarath_las_header = gujarath_las.header
# print(list(gujarath_las_header.point_format.dimension_names))

In [61]:
# las_x = gujarath_las.X
# las_y = gujarath_las.Y
# las_z = gujarath_las.Z
# las_rgb = gujarath_las.red, gujarath_las.green, gujarath_las.blue


In [62]:
def extract_dtm_from_las(las_file, output_path, outfile_name="output", chunk_size=3_000_000):

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    r_files, g_files, b_files, z_files = [], [], [], []

    with laspy.open(las_file) as reader:

        header = reader.header
        chunk_id = 0

        for points in reader.chunk_iterator(chunk_size):

            chunk_id += 1

            temp_las = output_path / f"chunk_{chunk_id}.las"

            # Write chunk
            with laspy.open(temp_las, mode="w", header=header) as writer:
                writer.write_points(points)

            # Output files
            output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
            output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
            output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
            output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"

            # PDAL pipeline
            pdal_pipeline = {
                "pipeline": [
                    {"type": "readers.las", "filename": str(temp_las)},
                    # {"type": "filters.smrf"},
                    {"type": "filters.range", "limits": "Classification[2:2]"},  # ground only

                    {"type": "writers.gdal",
                     "filename": str(output_r),
                     "dimension": "Red",
                     "resolution": 0.09,
                     "output_type": "mean",
                     "data_type": "float32"},

                    {"type": "writers.gdal",
                     "filename": str(output_g),
                     "dimension": "Green",
                     "resolution": 0.09,
                     "output_type": "mean",
                     "data_type": "float32"},

                    {"type": "writers.gdal",
                     "filename": str(output_b),
                     "dimension": "Blue",
                     "resolution": 0.09,
                     "output_type": "mean",
                     "data_type": "float32"},

                    {"type": "writers.gdal",
                     "filename": str(output_z),
                     "dimension": "Z",
                     "resolution": 0.09,
                     "output_type": "mean",
                     "data_type": "float32"}
                ]
            }

            pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
            pipeline.execute()

            r_files.append(str(output_r))
            g_files.append(str(output_g))
            b_files.append(str(output_b))
            z_files.append(str(output_z))

            print(f"Chunk {chunk_id} processed")

            temp_las.unlink()

    # -----------------------------
    # STEP 1: Mosaic each band
    # -----------------------------

    def mosaic(files, vrt_path, out_path):
        gdal.BuildVRT(str(vrt_path), files)
        gdal.Translate(str(out_path), str(vrt_path))

    mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
    mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
    mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
    mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

    mosaic(r_files, output_path / "R.vrt", mosaic_r)
    mosaic(g_files, output_path / "G.vrt", mosaic_g)
    mosaic(b_files, output_path / "B.vrt", mosaic_b)
    mosaic(z_files, output_path / "Z.vrt", mosaic_z)

    print("Per-band mosaics created")

    # -----------------------------
    # STEP 2: Create TRUE RGB (ONLY 3 bands)
    # -----------------------------

    final_rgb = output_path / f"{outfile_name}_RGB.tif"
    vrt_rgb = output_path / "rgb_stack.vrt"

    vrt_options = gdal.BuildVRTOptions(separate=True)

    # ONLY R, G, B here
    gdal.BuildVRT(
        str(vrt_rgb),
        [str(mosaic_r), str(mosaic_g), str(mosaic_b)],
        options=vrt_options
    )

    gdal.Translate(
        str(final_rgb),
        str(vrt_rgb),
        creationOptions=["TILED=YES", "COMPRESS=LZW", "PHOTOMETRIC=RGB"]
    )

    print("Final RGB raster saved at:", final_rgb)

    # -----------------------------
    # STEP 3: Save Z separately (DTM)
    # -----------------------------

    final_z = output_path / f"{outfile_name}_DTM.tif"

    gdal.Translate(
        str(final_z),
        str(mosaic_z),
        creationOptions=["TILED=YES", "COMPRESS=LZW"]
    )

    print("Final DTM (Z) saved at:", final_z)

In [63]:
def extract_dtm_from_las(las_file, output_path, outfile_name="output", chunk_size=1_000_000):

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    r_files, g_files, b_files, z_files = [], [], [], []

    with laspy.open(las_file) as reader:
        header   = reader.header
        chunk_id = 0

        for points in reader.chunk_iterator(chunk_size):
            chunk_id += 1

            temp_las = output_path / f"chunk_{chunk_id}.las"
            with laspy.open(temp_las, mode="w", header=header) as writer:
                writer.write_points(points)

            output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
            output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
            output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
            output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"

            # FIX: one pipeline per dimension
            for dimension, out_file in [
                ("Red",   output_r),
                ("Green", output_g),
                ("Blue",  output_b),
                ("Z",     output_z),
            ]:
                pipeline_def = {
                    "pipeline": [
                        {"type": "readers.las", "filename": str(temp_las)},
                        {"type"      : "filters.smrf",
                         "window"    : 5,
                         "slope"     : 0.15,
                         "threshold" : 0.5,
                         "scalar"    : 1.25,
                         "cut"       : 3.0
                        },
                        {"type": "filters.range", "limits": "Classification[2:2]"},
                        {
                            "type"       : "writers.gdal",
                            "filename"   : str(out_file),
                            "dimension"  : dimension,
                            "resolution" : 0.09,
                            "output_type": "mean",
                            "data_type"  : "float32"
                        }
                    ]
                }
                pipeline = pdal.Pipeline(json.dumps(pipeline_def))
                pipeline.execute()

            r_files.append(str(output_r))
            g_files.append(str(output_g))
            b_files.append(str(output_b))
            z_files.append(str(output_z))

            print(f"Chunk {chunk_id} processed — R, G, B, Z written")
            temp_las.unlink()

    # -----------------------------
    # STEP 1: Mosaic each band
    # -----------------------------
    def mosaic(files, vrt_path, out_path):
        gdal.BuildVRT(str(vrt_path), files)
        gdal.Translate(str(out_path), str(vrt_path))

    mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
    mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
    mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
    mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

    mosaic(r_files, output_path / "R.vrt", mosaic_r)
    mosaic(g_files, output_path / "G.vrt", mosaic_g)
    mosaic(b_files, output_path / "B.vrt", mosaic_b)
    mosaic(z_files, output_path / "Z.vrt", mosaic_z)

    print("Per-band mosaics done")

    # verify Z mosaic actually has data before stacking
    with rasterio.open(mosaic_z) as src:
        z_data = src.read(1)
        valid  = z_data[z_data != src.nodata]
        print(f"Z mosaic — valid pixels: {len(valid)}, range: {valid.min():.2f} → {valid.max():.2f}")

    # -----------------------------
    # STEP 2: Stack RGBZ (4 bands)
    # -----------------------------
    final_rgbz = output_path / f"{outfile_name}_RGBZ.tif"
    vrt_rgbz   = output_path / "rgbz_stack.vrt"

    gdal.BuildVRT(
        str(vrt_rgbz),
        [str(mosaic_r), str(mosaic_g), str(mosaic_b), str(mosaic_z)],
        options=gdal.BuildVRTOptions(separate=True)
    )

    gdal.Translate(
        str(final_rgbz),
        str(vrt_rgbz),
        creationOptions=["TILED=YES", "COMPRESS=LZW", "INTERLEAVE=PIXEL"]
    )

    # Verify final output
    with rasterio.open(final_rgbz) as src:
        print(f"\nFinal band count : {src.count}")       # must be 4
        for i in range(1, src.count + 1):
            band = src.read(i)
            print(f"  Band {i} range: {np.nanmin(band):.2f} → {np.nanmax(band):.2f}")

    print(f"\nFinal RGBZ saved: {final_rgbz}")

In [64]:
extract_dtm_from_las(gujarath_data, out_dir, "gujarath_dtm")

Chunk 1 processed — R, G, B, Z written
Chunk 2 processed — R, G, B, Z written
Chunk 3 processed — R, G, B, Z written
Chunk 4 processed — R, G, B, Z written
Chunk 5 processed — R, G, B, Z written
Chunk 6 processed — R, G, B, Z written
Chunk 7 processed — R, G, B, Z written
Chunk 8 processed — R, G, B, Z written
Chunk 9 processed — R, G, B, Z written
Chunk 10 processed — R, G, B, Z written
Chunk 11 processed — R, G, B, Z written
Chunk 12 processed — R, G, B, Z written
Chunk 13 processed — R, G, B, Z written
Chunk 14 processed — R, G, B, Z written
Chunk 15 processed — R, G, B, Z written
Chunk 16 processed — R, G, B, Z written
Chunk 17 processed — R, G, B, Z written
Chunk 18 processed — R, G, B, Z written
Chunk 19 processed — R, G, B, Z written
Chunk 20 processed — R, G, B, Z written
Chunk 21 processed — R, G, B, Z written
Chunk 22 processed — R, G, B, Z written
Chunk 23 processed — R, G, B, Z written
Chunk 24 processed — R, G, B, Z written
Chunk 25 processed — R, G, B, Z written
Chunk 26 

In [65]:
with laspy.open(gujarath_data) as reader:
        header = reader.header
        xmin, ymin, _ = header.mins
        xmax, ymax, _ = header.maxs

        bounds = f"([{xmin},{xmax}],[{ymin},{ymax}])"

print("Using global bounds:", bounds)

Using global bounds: ([258417.101,259517.34100000001],[2534666.112,2535568.722])


In [66]:
def extract_dsm_rgbz_from_las(las_file, output_path, outfile_name="output", chunk_size=3_000_000):

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    r_files, g_files, b_files, z_files = [], [], [], []

    # -----------------------------
    # STEP 0: Get GLOBAL bounds
    # -----------------------------
    with laspy.open(las_file) as reader:
        header = reader.header
        xmin, ymin, _ = header.mins
        xmax, ymax, _ = header.maxs

    bounds = f"([{xmin},{xmax}],[{ymin},{ymax}])"
    print("Using global bounds:", bounds)

    # -----------------------------
    # STEP 1: Process chunks
    # -----------------------------
    with laspy.open(las_file) as reader:

        chunk_id = 0

        for points in reader.chunk_iterator(chunk_size):

            chunk_id += 1

            temp_las = output_path / f"chunk_{chunk_id}.las"

            with laspy.open(temp_las, mode="w", header=header) as writer:
                writer.write_points(points)

            output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
            output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
            output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
            output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"

            # ✅ NO SMRF → DSM
            for dimension, out_file in [
                ("Red", output_r),
                ("Green", output_g),
                ("Blue", output_b),
                ("Z", output_z),
            ]:
                pipeline_def = {
                    "pipeline": [
                        {"type": "readers.las", "filename": str(temp_las)},
                        {
                            "type": "writers.gdal",
                            "filename": str(out_file),
                            "dimension": dimension,
                            "resolution": 0.09,
                            "output_type": "max",   # 🔥 DSM = max height
                            "data_type": "float32",
                            "bounds": bounds       # ✅ FIX HERE
                        }
                    ]
                }

                pipeline = pdal.Pipeline(json.dumps(pipeline_def))
                pipeline.execute()

            r_files.append(str(output_r))
            g_files.append(str(output_g))
            b_files.append(str(output_b))
            z_files.append(str(output_z))

            print(f"Chunk {chunk_id} processed (DSM)")

            temp_las.unlink()

    # -----------------------------
    # STEP 2: Mosaic
    # -----------------------------
    def mosaic(files, vrt_path, out_path):
        gdal.BuildVRT(str(vrt_path), files)
        gdal.Translate(str(out_path), str(vrt_path))

    mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
    mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
    mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
    mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

    mosaic(r_files, output_path / "R.vrt", mosaic_r)
    mosaic(g_files, output_path / "G.vrt", mosaic_g)
    mosaic(b_files, output_path / "B.vrt", mosaic_b)
    mosaic(z_files, output_path / "Z.vrt", mosaic_z)

    print("Mosaics done")

    # -----------------------------
    # STEP 3: Stack RGBZ
    # -----------------------------
    final_rgbz = output_path / f"{outfile_name}_RGBZ.tif"
    vrt_rgbz = output_path / "rgbz_stack.vrt"

    gdal.BuildVRT(
        str(vrt_rgbz),
        [str(mosaic_r), str(mosaic_g), str(mosaic_b), str(mosaic_z)],
        options=gdal.BuildVRTOptions(separate=True)
    )

    gdal.Translate(
        str(final_rgbz),
        str(vrt_rgbz),
        creationOptions=["TILED=YES", "COMPRESS=LZW", "INTERLEAVE=PIXEL"]
    )

    # -----------------------------
    # STEP 4: Validate
    # -----------------------------
    with rasterio.open(final_rgbz) as src:
        
        print(f"\nFinal band count: {src.count}")
        
        for i in range(1, src.count + 1):
            band = src.read(i)
            print(f"Band {i} range: {np.nanmin(band):.2f} → {np.nanmax(band):.2f}")

    print("\nDSM RGBZ saved at:", final_rgbz)

In [67]:
[os.path.join(out_dir,i) for i in os.listdir(out_dir) if i.endswith('RGBZ.tif')]

['/home/ajai-krishna/work/GEO_AI/outputs/subsets/gujarath_dtm_RGBZ.tif']

In [68]:
# extract_dsm_rgbz_from_las(gujarath_data,output_training_tif,"gujarath_dsm")

In [74]:
input_estimation_rgbz = [os.path.join(out_dir, i) for i in os.listdir(out_dir) if i.endswith('RGBZ.tif')][0]
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

output_estimation_rgbz = os.path.join(out_dir,"RGB_filled.tif")

In [75]:
fill_nodata_knn_chunkwise(input_estimation_rgbz,output_estimation_rgbz)

Raster size  : 10027 x 12225
Bands        : 4
Nodata value : -9999.0
Chunk size   : 1024
Overlap      : 64px
Total chunks : 120

Chunk [1/120] y=0 x=0 → filled 1,320,645 pixels
Chunk [2/120] y=0 x=1024 → filled 1,016,373 pixels
Chunk [3/120] y=0 x=2048 → filled 755,548 pixels
Chunk [4/120] y=0 x=3072 → filled 576,005 pixels
Chunk [5/120] y=0 x=4096 → filled 284,718 pixels
Chunk [6/120] y=0 x=5120 → filled 857,403 pixels
Chunk [7/120] y=0 x=6144 → all nodata skipped
Chunk [8/120] y=0 x=7168 → all nodata skipped
Chunk [9/120] y=0 x=8192 → all nodata skipped
Chunk [10/120] y=0 x=9216 → all nodata skipped
Chunk [11/120] y=0 x=10240 → all nodata skipped
Chunk [12/120] y=0 x=11264 → all nodata skipped
Chunk [13/120] y=1024 x=0 → filled 665,984 pixels
Chunk [14/120] y=1024 x=1024 → filled 205,410 pixels
Chunk [15/120] y=1024 x=2048 → filled 51,035 pixels
Chunk [16/120] y=1024 x=3072 → filled 154,318 pixels
Chunk [17/120] y=1024 x=4096 → filled 279,599 pixels
Chunk [18/120] y=1024 x=5120 → fil

In [73]:
fill_smrf_nodata(input_tiff_path, os.path.join(output_translate_dir, "gujarath_dtm_RGBZ_filled.tif"))

NameError: name 'fill_smrf_nodata' is not defined

In [72]:
dtm = rd.LoadGDAL(input_tiff_path)
filled = rd.FillDepressions(dtm)
rd.SaveGDAL(output_rd, filled)

NameError: name 'input_tiff_path' is not defined